# §39 step 2 — **K5dfiACC** and **L0dfiACC**: K5dfi's / L0dfi's recipes trained on ACCIDENT, read on TAD

Rule: `core/docs/final/KIM_RUN.md` **§39**, written before any number. Run **after** `accident_ingest.ipynb` (caches + `REPORTS/accident_dedup*`), and after the dedup has been read. PRELIMINARY (n = 1, s2024). **K5f stays the reference and the main model whatever the numbers (§39 N).**

| row | recipe | training data | input | here |
|---|---|---|---|---|
| **K5dfi** | §34 | Nexar_window | A2 V2-S bake | read only (control of P1) |
| **K5dfiACC** | K5dfi's | **ACCIDENT_window** | ACCIDENT A2 bake (Nexar's statistics) | train + read |
| **L0dfi** | §36 (KIM off, motion off) | Nexar_window | plain CLIP s8 | TAD read only (control of P2) |
| **L0dfiACC** | L0dfi's | **ACCIDENT_window** | plain CLIP s8 | train + read |
| K5f | reference | Nexar_window | A2 | read only (printed) |

**P1 (primary): TAD macro, K5dfiACC − K5dfi. P2: TAD macro, L0dfiACC − L0dfi.** Everything else printed. TAD micro is never evidence for or against H-dom (ACCIDENT has no normal video).

| step | what | GPU | estimate |
|---|---|---|---|
| 1-4 | setup, deps, helpers, preflight (K5dfi, L0dfi, K5f, bakes, ACCIDENT caches, dedup) | no | 3-5 min |
| 5 | tests — **raises if red** | no | 2 min |
| 6 | `ACCIDENT_window` (dedup exclusions), A2 bake with Nexar's statistics, plain s8, KNN, same-CLIP check | no | 10-20 min |
| 7 | train K5dfiACC, L0dfiACC (≈ 1,740 steps each) + guards | **yes** | 2 × K5dfi |
| 8 | reads: TAD, DoTA-CAP-dev, DADA, DAD (K5dfiACC); TAD, DoTA-CAP-dev, DADA (L0dfiACC); TAD (L0dfi); ACCIDENT val (both new rows) | yes | 1-1.5 h |
| 9 | tables -> **paste `REPORTS/k5dfiacc_report.md` to Claude** | no | 2 min |

**Watch the training log: past 2 s/step, stop and paste it.** A re-run skips what is on Drive. Thesis / Thesis-V2 are only read; everything new goes to `Thesis-Final/data/ACCIDENT_window/`, `Thesis-Final/cache/` and `Thesis-Final/outputs/kim_accident/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip (DoTA_s1_ncc, DADA2000_orig, TAD_ncc)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window (defs), A2 bakes
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5dfi / L0dfi / K5f, ACCIDENT caches; every output here
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 39. K5dfiACC' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §39 -- re-upload kat-vad/'
for tool in ('accident_build', 'accident_extract', 'accident_dedup'):
    assert (REPO / f'core/tools/{tool}.py').is_file(), f'no core/tools/{tool}.py -- re-upload kat-vad/'
assert (REPO / 'core/splits/accident/ACCIDENT_MANIFEST.json').is_file(), 'no frozen ACCIDENT split -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

# User overrides of the dedup (§39 D), recorded in KIM_RUN.md §39 BEFORE training: YouTube sources to keep
KEEP_SOURCES = []
# §39 D' resolution (2026-11-22, user): the percentile dedup STOPPED (tau 0.7911 flagged 52.2 %); the 88 clips with TAD
# top-1 >= 0.87 were read by eye on a score-sorted sheet (REPORTS/high/): none is the same camera -> nothing is excluded.
# False = the original rule (status must be OK, REPORTS/accident_dedup_exclude_ids.txt applies, minus KEEP_SOURCES).
DEDUP_NO_EXCLUSION = True

K5       = constants.KIM_MAIN_ARM                       # "K5": names run / eval sub-folders only
K5F      = constants.KIM_REFERENCE_ARM
K5DFI, L0 = constants.KIM_DVS_IGNORE_ROW, constants.KIM_LAGOVAD_NEXAR_ROW
K5A, L0A = constants.KIM_ACCIDENT_ROW, constants.KIM_ACCIDENT_L0_ROW
NEW_ROWS = (K5A, L0A)
SEED     = constants.KIM_SEED
ENC      = constants.V5_ENCODER
OFF      = constants.V2_OFF
IGN      = constants.DVS_ANCHOR_MODE_IGNORE
ACCNAME  = constants.ACCIDENT_DATASET
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
STRIDE_B = constants.V2_E1_STRIDE_BC
TAD_NAME = constants.TAD_DATASET
DAD_NAME, DAD_STRIDE, DAD_SET = constants.DAD_DATASET, constants.DAD_STRIDE, constants.DAD_OUTPUT_SET
OUT_F    = constants.OUTPUT_ROOT

V2_CACHE = Path(DRIVE_V2) / 'cache'
WINDOW   = Path(DRIVE_V2) / 'data' / f'{constants.NEXAR_DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'  # defs of every Nexar read
A2       = V2_CACHE / 'v2' / 'A2_S'
A2_NEXAR = A2 / 'Nexar_s8'                                    # the statistics the ACCIDENT rows are baked with (§39 B)
A2_DADA  = A2 / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = A2 / f'{CAP_DEV}_s3_from_s1_nexar'
A2_TAD   = A2 / f'{TAD_NAME}_s8_nexar'
A2_DAD   = constants.CACHE_ROOT / 'v2' / 'A2_S' / f'{DAD_NAME}_s{DAD_STRIDE}_nexar'
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
T2       = constants.DATA_ROOT / T2NAME
T2_CLIP  = Path(DRIVE) / 'cache' / 'clip' / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
TAD      = constants.DATA_ROOT / TAD_NAME
TAD_NCC  = Path(DRIVE) / 'cache' / 'clip' / 'TAD_ncc'
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
# rows read only
K5DFI_ROOT = OUT_F / constants.KIM_DVS_IGNORE_OUTPUT_DIRNAME / K5DFI
L0_ROOT    = OUT_F / constants.KIM_LAGOVAD_NEXAR_OUTPUT_DIRNAME / L0
K5F_ROOT   = OUT_F / constants.KIM_FIXED_OUTPUT_DIRNAME / K5F
K5DFI_RUN, L0_RUN = K5DFI_ROOT / K5 / f's{SEED}', L0_ROOT / K5 / f's{SEED}'
# ACCIDENT (written by accident_ingest.ipynb, then here)
ACC      = Path(DRIVE_F) / 'data' / ACCNAME
META     = ACC / constants.ACCIDENT_METADATA_FILENAME
CLIP_S1  = constants.CLIP_CACHE_DIR / constants.ACCIDENT_CLIP_S1_DATASET
VIDEO_S8 = constants.VIDEO_CACHE_DIR / ENC / f'{constants.ACCIDENT_VIDEO_S8_DATASET}_squash'   # V2-S rows = CLIP s1[::8]
CLIP_S8  = constants.CLIP_CACHE_DIR / constants.ACCIDENT_CLIP_S8_DATASET
BAKE     = constants.CACHE_ROOT / 'v2' / 'A2_S' / constants.ACCIDENT_BAKE_DIRNAME
DATA     = Path(DRIVE_F) / 'data' / f'{ACCNAME}_{constants.ACCIDENT_CONSTRUCTION_WINDOW}'
KNN      = constants.CACHE_ROOT / 'knn' / DATA.name / constants.KNN_CACHE_FILENAME
ROOT     = OUT_F / constants.KIM_ACCIDENT_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
DEDUP    = REPORTS / f'{constants.ACCIDENT_DEDUP_STEM}.json'
EXCLUDE  = REPORTS / f'{constants.ACCIDENT_DEDUP_STEM}_exclude_ids.txt'
BUILD    = REPORTS / constants.ACCIDENT_BUILD_REPORT_FILENAME
LOCAL    = Path('/content/k5dfi_acc')
LOCAL.mkdir(parents=True, exist_ok=True)


def row_paths(row):
    run_dir = ROOT / row / K5 / f's{SEED}'
    return run_dir, run_dir / 'checkpoint_last.pt', ROOT / row / 'eval'


INPUTS = (('K5dfi run', K5DFI_RUN / 'verified.json'), ('K5dfi config', K5DFI_RUN / 'config.yaml'),
          ('L0dfi run', L0_RUN / 'verified.json'), ('L0dfi checkpoint', L0_RUN / 'checkpoint_last.pt'),
          ('L0dfi config', L0_RUN / 'config.yaml'),
          *[(f'K5dfi {s} read', K5DFI_ROOT / 'eval' / s / K5) for s in ('dota', 'dada', 'tad', DAD_SET)],
          *[(f'K5f {s} read', K5F_ROOT / 'eval' / s / K5) for s in ('dota', 'dada', 'tad', DAD_SET)],
          ('L0dfi DoTA-CAP-dev read', L0_ROOT / 'eval' / CAP_DEV / K5 / 'clip_scores.npz'),
          ('L0dfi DADA read', L0_ROOT / 'eval' / 'dada' / K5 / 'clip_scores.npz'),
          ('Nexar_window defs', WINDOW / constants.DEFS_FILENAME), ('Nexar A2 bake', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME),
          ('DADA A2 bake', A2_DADA), ('DoTA-CAP A2 bake', A2_DOTA), ('TAD A2 bake', A2_TAD), ('DAD A2 bake', A2_DAD),
          ('DoTA s1', DOTA_S1), ('T2 CLIP', T2_CLIP), ('TAD_ncc', TAD_NCC), ('T2 meta', T2 / constants.META_FILENAME),
          ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT), ('DoTA metadata', DOTA / 'metadata_val.json'),
          ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME), ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME),
          ('ACCIDENT metadata', META), ('ACCIDENT CLIP s1', CLIP_S1), ('ACCIDENT V2-S s8', VIDEO_S8),
          ('dedup report (run accident_ingest.ipynb)', DEDUP), ('dedup exclusions', EXCLUDE))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | runs / reads -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 7-8 need one'
df -h /content | tail -1

In [ ]:
import numpy as np  # noqa: E402
import torch  # noqa: E402
import yaml  # noqa: E402

from core.config import Config                       # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.accident_build import load_accident_split, read_metadata, source_of  # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    names = [f.name for f in src.iterdir() if f.is_file()] if ids is None else \
        [f'{v}.npy' for v in ids] + [m.name for p in ('*.json', '*.npz') for m in src.glob(p)]
    todo = [n for n in names if not (dst / n).exists()]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda n: shutil.copy2(src / n, dst / n), todo))
    print(f'  staged {src.name}: {len(todo)} files in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def on_drive(folder, min_bytes=1024):
    """Ids of every finished ``.npy`` in ``folder`` from ONE listing (save_array = tmp + rename)."""
    if not folder.is_dir():
        return set()
    with os.scandir(folder) as entries:
        return {e.name[:-4] for e in entries if e.name.endswith('.npy') and e.stat().st_size >= min_bytes}


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(run_dir, steps):
    rec, ckpt = run_dir / 'verified.json', run_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


DEFAULTS = _flat(Config().to_dict())


def config_diff(base, got):
    """Keys that differ; a key added after ``base`` trained may sit at its default."""
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k):
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out


# Preflight on small files only (C39)
dfi_cfg = _flat(yaml.safe_load((K5DFI_RUN / 'config.yaml').read_text()))
assert dfi_cfg['kim.enabled'] and dfi_cfg['v2.motion'] == ENC and dfi_cfg['loss.dvs_anchor_mode'] == IGN, 'K5DFI_RUN is not K5dfi'
assert dfi_cfg['dvs.theta'] == 0.7 and dfi_cfg['dvs.theta_ego'] == 0.85 and not dfi_cfg['data.is_egocentric'], 'not K5dfi'
l0_cfg = _flat(yaml.safe_load((L0_RUN / 'config.yaml').read_text()))
assert not l0_cfg['kim.enabled'] and l0_cfg['v2.motion'] == OFF and l0_cfg['loss.dvs_anchor_mode'] == IGN, 'L0_RUN is not L0dfi'
for name, run_dir in ((K5DFI, K5DFI_RUN), (L0, L0_RUN)):
    steps = json.loads((run_dir / 'verified.json').read_text())['step']
    assert is_verified(run_dir, steps) and metrics_rows(run_dir / 'metrics.jsonl')[-1]['global_step'] == steps, f'{name}: J10'
SHA = manifest(A2_NEXAR)['train_ids_sha1']
for name, d in (('Nexar', A2_NEXAR), ('DADA', A2_DADA), ('DoTA-CAP', A2_DOTA), ('TAD', A2_TAD), ('DAD', A2_DAD)):
    m = manifest(d)
    assert (m['crn'], m['motion'], m['train_ids_sha1']) == (OFF, ENC, SHA), f'{name} bake: {m}'
rows = read_metadata(META)
ACC_TRAIN = load_accident_split(constants.ACCIDENT_SPLIT_TRAIN)
ACC_VAL = load_accident_split(constants.ACCIDENT_SPLIT_VAL)
DADA_IDS = test_sources(T2)
CAP_IDS = sorted(load_split(CAP_DEV))
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(DADA_IDS) == 370 and len(TAD_IDS) == constants.V2_TADZ_VIDEOS and len(DAD_IDS) == constants.DAD_TEST_VIDEOS
from core.tools.accident_build import windowable  # noqa: E402
EXTRACTED = sorted(v for v in rows if windowable(rows[v]))      # accident_ingest skips clips shorter than 1 window
done = on_drive(CLIP_S1) & on_drive(VIDEO_S8)
assert set(EXTRACTED) <= done, f'{len(set(EXTRACTED) - done)} ACCIDENT clips not extracted: run accident_ingest.ipynb'
dedup = json.loads(DEDUP.read_text())
if DEDUP_NO_EXCLUSION:                  # §39 D' (read by eye, nothing excluded); the STOP status is the reason, not a defect
    assert dedup['status'] == constants.ACCIDENT_DEDUP_STOP, f"D' resolves a STOP, got {dedup['status']}: use the original rule"
    assert not KEEP_SOURCES, 'KEEP_SOURCES is meaningless when nothing is excluded'
    EXCLUDED = []
else:
    assert dedup['status'] == constants.ACCIDENT_DEDUP_OK, f"dedup status {dedup['status']}: paste REPORTS/accident_dedup.md to Claude"
    unknown = sorted(set(KEEP_SOURCES) - set(dedup['excluded_sources']))
    assert not unknown, f'KEEP_SOURCES not excluded by the dedup: {unknown}'
    EXCLUDED = sorted(v for v in EXCLUDE.read_text().split() if source_of(v) not in set(KEEP_SOURCES))
print(f'preflight OK: K5dfi / L0dfi verified | ACCIDENT {len(rows)} clips extracted, split {len(ACC_TRAIN)} / {len(ACC_VAL)} | '
      f'dedup tau {dedup["tau"]:.4f}: {len(EXCLUDED)} clips excluded ({len(KEEP_SOURCES)} sources kept by the user)')

## 5. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_accident.py',
     'core/tests/test_dvs_dataset.py', 'core/tests/test_kim.py', 'core/tests/test_tad_category_read.py'])

## 6. `ACCIDENT_window`, the A2 bake (Nexar's statistics), plain s8, KNN, and the same CLIP frames for both rows

In [ ]:
L_X = stage(CLIP_S1, LOCAL / 'stage' / 'acc_x', EXTRACTED)
excl_file = write_ids(LOCAL / 'ids' / 'exclude.txt', EXCLUDED)
if not (DATA / constants.LABELS_TRAIN_FILENAME).exists() or not BUILD.exists():
    local = LOCAL / 'corpus' / DATA.name
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.accident_build', 'corpus', '--metadata', META, '--clip-s1-dir', L_X,
         '--exclude-ids', excl_file, '--out-dir', local, '--report', LOCAL / 'build_report.json'], log=LOCAL / 'build.log')
    publish(local, DATA)
    publish(LOCAL / 'build_report.json', BUILD)
    flush_drive()
build = json.loads(BUILD.read_text())
info, PLAN = build['construction'], build['plan']
from core.data.v2_splits import lines_sha1  # noqa: E402
assert build['excluded_ids_sha1'] == lines_sha1(EXCLUDED), 'ACCIDENT_window was built with other exclusions: delete it and re-run'
labels = json.loads((DATA / constants.LABELS_TRAIN_FILENAME).read_text())
windows = json.loads((DATA / constants.WINDOWS_FILENAME).read_text())
train_src = {windows[w]['source'] for w in labels}
assert not train_src & set(ACC_VAL) and not train_src & set(EXCLUDED), 'a val / excluded clip trains'
x_rows = {v: int(np.load(L_X / f'{v}.npy', mmap_mode='r').shape[0]) for v in train_src}
from core.tools.accident_build import cut_frames  # noqa: E402
past = [w for w in labels if windows[w]['end'] > -(-cut_frames(rows[windows[w]['source']], x_rows[windows[w]['source']]) // 8)]
assert not past, f'{len(past)} windows reach past t_impact + 2 s'
assert (DATA / constants.DEFS_FILENAME).read_text() == (WINDOW / constants.DEFS_FILENAME).read_text(), 'defs differ from Nexar_window'
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']
print(f"ACCIDENT_window: {info['train_abnormal']} positive / {info['train_items']} windows from {len(train_src)} clips | "
      f"val {info['val_windows']} windows ({info['val_two_class_windows']} two-class) | plan {PLAN}")

ACC_IDS = sorted({windows[w]['source'] for w in windows})           # train + val sources with a window
ids_file = write_ids(LOCAL / 'ids' / 'acc_bake.txt', ACC_IDS)
if not set(ACC_IDS) <= on_drive(CLIP_S8):                        # plain CLIP s8 = s1[::8] first ...
    local = LOCAL / 'clip_s8'
    run([sys.executable, '-m', 'core.tools.nexar_build', 'subsample', '--clip-s1-dir', L_X, '--ids-file', ids_file,
         '--stride', constants.FRAME_STRIDE, '--out-dir', local])
    publish(local, CLIP_S8)
    flush_drive()
L_S8 = stage(CLIP_S8, LOCAL / 'stage' / 'acc_s8', ACC_IDS)
if not (BAKE / constants.V2_INPUT_MANIFEST_FILENAME).exists() or not set(ACC_IDS) <= on_drive(BAKE):
    # ... then the bake reads s8 CLIP + the s8 V2-S cache at stride 1 (= Nexar's s1 bake at --stride 8)
    L_U = stage(VIDEO_S8, LOCAL / 'stage' / 'acc_u', ACC_IDS)
    bad = [v for v in ACC_IDS if np.load(L_U / f'{v}.npy', mmap_mode='r').shape[0] != np.load(L_S8 / f'{v}.npy', mmap_mode='r').shape[0]]
    assert not bad, f'{len(bad)} clips: V2-S s8 rows != CLIP s8 rows: {bad[:5]}'
    L_STATS = stage(A2_NEXAR, LOCAL / 'stage' / 'nexar_stats', [])    # manifest + statistics only
    local = LOCAL / 'bake'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', L_STATS, '--clip-dir', L_S8,
         '--video-dir', L_U, '--ids-file', ids_file, '--stride', 1, '--out-dir', local])
    publish(local, BAKE)
    flush_drive()
m = manifest(BAKE)
assert (m['crn'], m['motion'], m['train_ids_sha1']) == (OFF, ENC, SHA) and m['baked_ids'] == len(ACC_IDS), f'ACCIDENT bake: {m}'
L_BAKE = stage(BAKE, LOCAL / 'stage' / 'acc_bake', ACC_IDS)
run([sys.executable, '-m', 'core.tools.zs_lagovad_readout', 'check-inputs', '--plain', L_S8, '--bake', L_BAKE,
     '--ids-file', ids_file, '--plain-stride', 1])
L_DATA = stage(DATA, LOCAL / 'stage' / 'data')
if not KNN.exists():
    local = LOCAL / 'knn' / KNN.name
    run([sys.executable, '-m', 'core.data.knn_cache', '--data-dir', L_DATA, '--dataset', ACCNAME,
         '--clip-dir', L_S8, '--output', local])
    publish(local, KNN)
L_KNN = stage(KNN, LOCAL / 'stage' / 'knn' / KNN.name)
print(f'bake {BAKE.name}: {m["baked_ids"]} clips, Nexar statistics ({SHA[:12]}) | plain s8 == bake CLIP columns | KNN {KNN}')

## 7. Train **K5dfiACC** and **L0dfiACC** (§39 A1)

Each = its parent's `--set` list with `data.dataset=ACCIDENT` and the epochs of the plan. Guards (§39 G): J10 at the plan's steps, config diff to the parent ⊆ {`data.dataset`, `train.num_epochs`}, last-epoch `mul_mil` > 0.01. **Watch the log: past 2 s/step, stop and paste it.**

In [ ]:
K5_BASE = ['train.stage=2', f'train.seed={SEED}', 'train.amp=true',
           f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}', 'kip.enabled=false', f'loss.dvs_anchor_mode={IGN}',
           'data.is_egocentric=false', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
           'dvs.theta=0.7', 'dvs.theta_ego=0.85']
SETS = {K5A: [*K5_BASE, f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}', 'kim.enabled=true',
              'kim.prototypes=true', 'kim.edge_mask=true', 'kim.seam_mask=true'],
        L0A: [*K5_BASE, 'kim.enabled=false', f'v2.motion={OFF}', 'model.motion_dim=0']}
PARENT = {K5A: (K5DFI, dfi_cfg), L0A: (L0, l0_cfg)}
INPUT = {K5A: L_BAKE, L0A: L_S8}
ALLOWED = {'data.dataset', 'train.num_epochs'}


def check_run(row):
    run_dir, ckpt, _ = row_paths(row)
    mrows = metrics_rows(run_dir / 'metrics.jsonl')
    assert ckpt_step(ckpt) == mrows[-1]['global_step'] == STEPS, f'{row}: J10'
    got = _flat(yaml.safe_load((run_dir / 'config.yaml').read_text()))
    parent, base = PARENT[row]
    diff = config_diff(base, got)
    assert set(diff) <= ALLOWED and diff['data.dataset'][1] == ACCNAME, f'{row} vs {parent}: {diff}'
    assert got['train.num_epochs'] == EPOCHS, got['train.num_epochs']
    last = [r for r in mrows if r['epoch'] == mrows[-1]['epoch']]
    mul_last = sum(r['mul_mil'] for r in last) / len(last)
    mil_last = sum(r['mil'] for r in last) / len(last)
    assert mul_last > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{row}: last-epoch mul_mil {mul_last:.4f}'
    ref = metrics_rows((K5DFI_RUN if row == K5A else L0_RUN) / 'metrics.jsonl')
    ref_last = [r for r in ref if r['epoch'] == ref[-1]['epoch']]
    print(f'{row}: config diff to {parent} {diff} | last-epoch mil {mil_last:.4f} ({parent} '
          f"{sum(r['mil'] for r in ref_last) / len(ref_last):.4f}) | mul_mil {mul_last:.4f} > {constants.KIM_FIXED_MUL_MIL_FLOOR}")


def train_row(row):
    run_dir, ckpt, _ = row_paths(row)
    if is_verified(run_dir, STEPS):
        print(f'{row} s{SEED}: verified on Drive -- not retrained')
        check_run(row)
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / row
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    sets = [*SETS[row], f'data.dataset={ACCNAME}', f'train.num_epochs={EPOCHS}']
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in sets for x in ('--set', s)], '--data-dir', L_DATA,
         '--clip-dir', INPUT[row], '--output-dir', local, '--knn-cache', L_KNN, *resume], log=LOCAL / f'train_{row}.log')
    last = metrics_rows(local / 'metrics.jsonl')[-1]
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{row}: J10 locally {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, run_dir / name)
    flush_drive()
    assert ckpt_step(ckpt) == STEPS, f'{row}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': ckpt.stat().st_size}))
    publish(local / 'verified.json', run_dir / 'verified.json')
    print(f'{row}: trained in {(time.time() - t0) / 60:.0f} min')
    check_run(row)


for row in NEW_ROWS:
    train_row(row)

## 8. Reads (K5dfi's protocols and bakes; plain CLIP for the L0 rows)

In [ ]:
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_TAD = LOCAL / 'TAD_meta'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every Nexar row read with
    shutil.copy2(f, L_DEFS / f.name)
L_LAB = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')
L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS)


def read_evaluate(row, name, dataset, data_dir, clip_dir, n_items, out, ckpt, raw=True):
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / row / name
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt, '--set', f'data.dataset={dataset}',
             '--data-dir', data_dir, '--clip-dir', clip_dir, '--score-norm', 'auto', '--save-scores',
             '--output-dir', local], log=LOCAL / f'{row}_{name}.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['num_videos'] == n_items and (not raw or r['score_norm'] == 'none'), r
    print(f"{row} {name}: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")
    return r


def read_dota(row, ckpt, out, input_dir, plain):
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dota'
        shutil.rmtree(local, ignore_errors=True)
        stride = ['--stride', STRIDE_B] if plain else []
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ckpt, '--input-dir', input_dir,
             '--s1-dir', L_DOTA_S1, '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--data-dir', L_LAB, '--split', CAP_DEV, *stride, '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{row}_dota.log')
        publish(local, out)
    print(f"{row} DoTA-CAP-dev: macro {json.loads((out / 'protocol_b_readout.json').read_text())['macro']['mean']:.4f}")


def read_dada(row, ckpt, out, input_dir):
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dada'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', ckpt, '--t2-dir', L_T2,
             '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', input_dir,
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'{row}_dada.log')
        publish(local, out)
    print(f"{row} DADA whole: macro {json.loads((out / 'dada_readout.json').read_text())['macro']['mean']:.4f}")


TAD_IN = {'bake': stage(A2_TAD, LOCAL / 'stage' / 'tad_bake', TAD_IDS), 'plain': stage(TAD_NCC, LOCAL / 'stage' / 'tad_ncc', TAD_IDS)}
VAL = {}
# K5dfiACC: K5dfi's four sets on K5dfi's bakes, + ACCIDENT val (printed)
_, ckpt, ev = row_paths(K5A)
read_evaluate(K5A, 'tad', TAD_NAME, L_TAD, TAD_IN['bake'], len(TAD_IDS), ev / 'tad' / K5 / f's{SEED}', ckpt)
read_dota(K5A, ckpt, ev / 'dota' / K5, stage(A2_DOTA, LOCAL / 'stage' / 'cap_bake', CAP_IDS), plain=False)
read_dada(K5A, ckpt, ev / 'dada' / K5, stage(A2_DADA, LOCAL / 'stage' / 'dada_bake', DADA_IDS))
read_evaluate(K5A, 'dad', DAD_NAME, stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta'), stage(A2_DAD, LOCAL / 'stage' / 'dad_bake', DAD_IDS),
              len(DAD_IDS), ev / DAD_SET / K5 / f's{SEED}', ckpt)
VAL[K5A] = read_evaluate(K5A, 'accident_val', ACCNAME, L_DATA, L_BAKE, info['val_windows'], ev / 'accident_val' / K5 / f's{SEED}', ckpt, raw=False)
flush_drive()
# L0dfiACC: plain CLIP inputs (L0dfi's)
_, ckpt, ev = row_paths(L0A)
read_evaluate(L0A, 'tad', TAD_NAME, L_TAD, TAD_IN['plain'], len(TAD_IDS), ev / 'tad' / K5 / f's{SEED}', ckpt)
read_dota(L0A, ckpt, ev / 'dota' / K5, L_DOTA_S1, plain=True)
read_dada(L0A, ckpt, ev / 'dada' / K5, stage(T2_CLIP, LOCAL / 'stage' / 'dada_clip', DADA_IDS))
VAL[L0A] = read_evaluate(L0A, 'accident_val', ACCNAME, L_DATA, L_S8, info['val_windows'], ev / 'accident_val' / K5 / f's{SEED}', ckpt, raw=False)
# L0dfi: TAD on plain TAD_ncc (it had no TAD read); written under this notebook's folder, L0dfi's own is untouched
read_evaluate(L0, 'tad', TAD_NAME, L_TAD, TAD_IN['plain'], len(TAD_IDS), ROOT / L0 / 'eval' / 'tad' / K5 / f's{SEED}', L0_RUN / 'checkpoint_last.pt')
flush_drive()

## 9. Tables -> **paste `REPORTS/k5dfiacc_report.md` to Claude**

`zs_arm_table` control **K5dfi** (rows K5dfiACC, K5f; four sets) and control **L0dfi** (row L0dfiACC; TAD, DoTA-CAP-dev, DADA), TAD by category for the four rows. **P1 = TAD macro K5dfiACC − K5dfi, P2 = TAD macro L0dfiACC − L0dfi.** Printed only (§39 N).

In [ ]:
local = LOCAL / 'report'
local.mkdir(parents=True, exist_ok=True)
EV = {K5DFI: K5DFI_ROOT / 'eval', K5F: K5F_ROOT / 'eval', K5A: ROOT / K5A / 'eval', L0A: ROOT / L0A / 'eval'}
reads = [a for row in (K5DFI, K5A, K5F) for s in ('dota', 'dada', 'tad', DAD_SET) for a in ('--read', s, row, EV[row] / s / K5)]
NAME = constants.KIM_ACCIDENT_TABLE_NAME
run([sys.executable, '-m', 'core.tools.zs_arm_table', '--control', K5DFI, '--seed', SEED, *reads, '--name', NAME, '--out-dir', local])
L0_READS = {'dota': L0_ROOT / 'eval' / CAP_DEV / K5, 'dada': L0_ROOT / 'eval' / 'dada' / K5, 'tad': ROOT / L0 / 'eval' / 'tad' / K5}
reads = [a for s, d in L0_READS.items() for a in ('--read', s, L0, d)]
reads += [a for s in L0_READS for a in ('--read', s, L0A, EV[L0A] / s / K5)]
NAME_L0 = constants.KIM_ACCIDENT_L0_TABLE_NAME
run([sys.executable, '-m', 'core.tools.zs_arm_table', '--control', L0, '--seed', SEED, *reads, '--name', NAME_L0, '--out-dir', local])
tad_dirs = {K5DFI: EV[K5DFI] / 'tad' / K5 / f's{SEED}', K5A: EV[K5A] / 'tad' / K5 / f's{SEED}',
            L0: L0_READS['tad'] / f's{SEED}', L0A: EV[L0A] / 'tad' / K5 / f's{SEED}'}
run([sys.executable, '-m', 'core.tools.tad_category_read', *[a for r, d in tad_dirs.items() for a in ('--read', r, d)],
     '--name', NAME, '--out-dir', local])

stem = lambda s, n: (local / f'{s}_{n}.md').read_text()
lines = ['# K5dfiACC / L0dfiACC (KIM_RUN.md §39), s2024, n = 1 — printed, decides nothing', '',
         f"ACCIDENT_window: {info['train_abnormal']} positive / {info['train_items']} train windows from "
         f"{info['train_sources_with_windows']} clips ({info['train_clips_after_dedup']} after dedup of {info['train_clips']}); "
         f"val {info['val_windows']} windows ({info['val_two_class_windows']} two-class). Dedup: tau {dedup['tau']:.4f} "
         f"({'STOP, resolved by §39 D′: top-1 >= 0.87 read by eye, no same camera' if DEDUP_NO_EXCLUSION else 'OK'}), "
         f"{len(EXCLUDED)} clips excluded, {len(KEEP_SOURCES)} sources kept by the user. Plan {PLAN}. "
         'Baked with Nexar A2 statistics (§39 B).', '',
         '## ACCIDENT val (in-domain, printed only)', '', '| row | macro | micro | AP | score norm |', '|---|---|---|---|---|']
lines += [f"| {r} | {v['auc_macro']:.4f} | {v['auc']:.4f} | {v['ap']:.4f} | {v['score_norm']} |" for r, v in VAL.items()]
lines += ['', '## Control K5dfi (P1 = TAD macro K5dfiACC − K5dfi)', '', stem(constants.ZS_ARM_TABLE_STEM, NAME), '',
          '## Control L0dfi (P2 = TAD macro L0dfiACC − L0dfi)', '', stem(constants.ZS_ARM_TABLE_STEM, NAME_L0), '',
          stem(constants.TAD_CATEGORY_STEM, NAME)]
(local / f'{constants.KIM_ACCIDENT_REPORT_STEM}.md').write_text('\n'.join(lines) + '\n')
publish(local, REPORTS)
print((REPORTS / f'{constants.KIM_ACCIDENT_REPORT_STEM}.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k5dfiacc_report.md to Claude')